# Task 2, Part B2: Linear (ELU+1) and Mixed Local/Dilated Sparse Attention on MQAR

Same architecture, optimiser and budget as B1; only the attention operator changes.
* **Linear:** causal ELU+1 kernel attention (chunked recurrence over S = sum phi(k) v^T and z = sum phi(k)),
  K in {4, 8, 16}, mixed regime.
* **Mixed local/dilated:** in every block heads 0-1 attend the preceding 64 positions, heads 2-3 attend 64 positions
  at offsets {0, 4, ..., 252}; K in {4, 16}, mixed regime. Implemented with an explicit dense boolean mask + SDPA
  (a correctness implementation, not an efficient sparse kernel; see notebook 4 for a gather implementation).
Compared on near/far accuracy, loss, runtime and the number of candidate positions per query.

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task2_attention_kv")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task2", need_public=False, need_prefix_banks=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from shared.src.io import load_json
from shared.src.plotting import save_figure
from task2_attention_kv.src import mqar_data as D
from task2_attention_kv.src.mqar_train import PROTOCOL, train_run, test_eval, run_name
from task2_attention_kv.src.mqar_model import MQARModel, n_params, attended_pairs, state_bytes, receptive_field

OUT = ctx.results / "mqar"
CACHE = ctx.checkpoints / "mqar_cache"
dev, test = D.load_frozen("development"), D.load_frozen("test")
RUNS = [("linear", "mixed", 4), ("linear", "mixed", 8), ("linear", "mixed", 16), ("dilated", "mixed", 4), ("dilated", "mixed", 16)]
SEEDS = PROTOCOL["seeds"]
print({k: n_params(MQARModel(k)) for k in ("dense", "linear", "dilated")})

In [ ]:
for seed in SEEDS:
    for kind, regime, k in RUNS:
        h = train_run(kind, k, regime, seed, dev, OUT, CACHE)
        print(f"{h['name']}: {h['train_time_s']:.0f}s final dev {h['dev'][-1]}")

In [ ]:
results = [test_eval(OUT, test, kind, regime, k, seed) for seed in SEEDS for kind, regime, k in RUNS]
rows = []
for r in results:
    for cell, v in r["cells"].items():
        k, reg = cell.split("_")
        rows.append(dict(model=r["kind"], train=f"{r['train_regime']} K{r['train_K']}", seed=r["seed"], test_K=int(k[1:]), test_regime=reg, **v))
df = pd.DataFrame(rows)
df.to_csv(OUT / "B2_test_cells.csv", index=False)
same_K = df[df.apply(lambda r: f"K{r.test_K}" in r.train, axis=1)]
same_K.groupby(["model", "train", "test_regime"])[["acc", "loss"]].agg(["mean", "std"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for kind, regime, k in RUNS:
    H = [load_json(OUT / "histories" / f"{run_name(kind, regime, k, s)}.json") for s in SEEDS]
    steps = [p["step"] for p in H[0]["train_loss"]]
    L = np.array([[p["loss"] for p in h["train_loss"]] for h in H])
    axes[0].plot(steps, L.mean(0), label=f"{kind} K{k}")
    axes[0].fill_between(steps, L.mean(0) - L.std(0), L.mean(0) + L.std(0), alpha=0.2)
    A = np.array([[p[f"K{k}_{regime}"] for p in h["dev"]] for h in H])
    axes[1].errorbar([p["step"] for p in H[0]["dev"]], A.mean(0), yerr=A.std(0), marker="o", capsize=3, label=f"{kind} K{k}")
axes[0].set_yscale("log"); axes[0].set_xlabel("update"); axes[0].set_ylabel("train CE")
axes[1].set_xlabel("update"); axes[1].set_ylabel("dev accuracy (mixed)"); axes[1].legend()
save_figure(ctx.figures, "mqar_B2_learning_curves")

## Accuracy by source-query distance bin (training K, mixed test regime)

In [ ]:
bins = ["acc_d0_32", "acc_d32_64", "acc_d64_128", "acc_d128_256"]
same_K[same_K.test_regime == "mixed"].groupby(["model", "train"])[bins].mean().round(3)

## Runtime, parameters, candidate positions per query (T = 256)

In [ ]:
rt = []
for kind, regime, k in RUNS:
    H = [load_json(OUT / "histories" / f"{run_name(kind, regime, k, s)}.json") for s in SEEDS]
    rt.append(dict(kind=kind, K=k, params=H[0]["n_params"], train_time_s=np.mean([h["train_time_s"] for h in H]),
                   peak_mem_mb=np.mean([h["peak_mem_mb"] or 0 for h in H]), **attended_pairs(kind, 256), **state_bytes(kind, 256),
                   receptive_field=receptive_field(kind)))
pd.DataFrame(rt)